In [0]:
#dbutils.fs.rm("/Volumes/data_landing/data_raw/city_time_series/_checkpoints", True)

# this will be uncommented for initial run and 
# for further runs this commented due to following reason
# The Checkpoint "Memory" (Most Likely): Auto Loader uses the checkpointLocation to track which files have been processed. If you ran this code once (even with errors or with a different schema) and it "successfully" acknowledged chunk3.json, it will never process that file again, even if the table is empty or deleted. ONLY UNCOMMENT FOR FIRST RUN OR TESTING and then COMMENT For IDEMPOTENCY.

**1. Architecture Overview**
Auto Loader is designed for incremental and idempotent data ingestion. It uses a mechanism called "Schema Inference" and "Checkpointing" to track which files have been processed.

Why we are using this specific approach:
- Memory Efficiency: By providing an explicit schema (sampling the first row), we prevent the Spark Driver from scanning the entire 511MB file, which would cause an OutOfMemoryError.
- Idempotency: The checkpointLocation ensures that if the job fails and restarts, Spark ignores files that were already successfully written to the Bronze table.
- Uniformity: Every column is cast to String to match existing COPY INTO tables, while load_dt is system-generated as a Timestamp.

**2. Prerequisites & Setup**
Before executing the code, ensure the following configurations are in place:

A. Directory Structure
Ensure your Unity Catalog Volumes are organized as follows:
- Landing (Source): /Volumes/data_landing/data_raw/<dataset_name>/chunks/chunk3.json
- Bronze (Metadata): /Volumes/data_bronze/bronze/checkpoints/<dataset_name>_json/

B. Cluster Settings
- Runtime: Databricks Runtime 13.3 LTS or higher.
- Permissions: Ensure you have READ VOLUME on the landing path and WRITE VOLUME on the bronze path.


In [0]:
import os
import json
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# --- 1. CONFIGURATION ---
# VOLUME_PREFIX: Spark-compatible path for reading from Unity Catalog Volumes
VOLUME_PREFIX    = "dbfs:/Volumes/data_landing/data_raw" 
DEST_CATALOG     = "data_bronze"
DEST_SCHEMA      = "bronze"

def process_json_stream(dataset_name, vol_path, target_table):
    """
    Ingests ONLY chunk3.json using Auto Loader. 
    Implements a 'Fresh-Start' checkpoint strategy to ensure 100% data delivery.
    """
    source_folder = f"{vol_path}/chunks"
    # checkpoint_path: Stores the state of the stream (which files have been seen)
    checkpoint_root = f"{vol_path}/_checkpoints"
    checkpoint_path = f"{checkpoint_root}/json_ingest_{dataset_name}"
    
    print(f"   - Initializing Auto Loader for {dataset_name}...")

    # --- 1. IDEMPOTENCY & CHECKPOINT CLEANUP ---
    # This block ensures we don't ingest the same data twice, but also recovers
    # if a checkpoint exists without the data actually being in the Delta table.
    try:
        table_exists = spark.catalog.tableExists(target_table)
        data_already_in_table = False
        
        if table_exists:
            # Check if this specific chunk has already been loaded
            data_already_in_table = spark.table(target_table).filter(F.col("source") == "chunk3.json").limit(1).count() > 0
        
        if data_already_in_table:
            print(f"   [IDEMPOTENT] chunk3.json already exists in {target_table}. Skipping.")
            return
        else:
            # If data is NOT in the table but a checkpoint exists, the checkpoint is 'stale'.
            # We must remove it so Auto Loader re-scans and ingests the file.
            print(f"   [CLEANUP] Removing stale checkpoint at: {checkpoint_path}")
            dbutils.fs.rm(checkpoint_path, True)
            
    except Exception as e:
        print(f"   [WARNING] Error during checkpoint/idempotency check: {str(e)}")

    # --- 2. SETUP AUTO LOADER (cloudFiles) ---
    # Auto Loader provides schema inference and efficient file tracking.
    stream_df = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", checkpoint_path) # Stores inferred schema
        .option("cloudFiles.inferColumnTypes", "false")      # Bronze requirement: keep as strings
        .option("pathGlobFilter", "chunk3.json")             # Strictly target the JSON chunk
        .option("multiline", "true")                         # Required for standard JSON arrays/objects
        .load(source_folder))

    # --- 3. TRANSFORM & METADATA ---
    # Prepare data for Bronze tier by standardizing types and adding lineage audit columns
    data_columns = [c for c in stream_df.columns if c.lower() not in ["load_dt", "source"]]

    final_df = stream_df.select("*", "_metadata.file_path").select(
        # Cast all business columns to string for Bronze stability
        *[F.col(c).cast("string") for c in data_columns],
        # Add Audit metadata for Data Quality tracking
        F.current_timestamp().alias("load_dt"), 
        # Extract filename from the internal file_path metadata for lineage
        F.element_at(F.split(F.col("file_path"), "/"), -1).alias("source")
    )

    # --- 4. WRITE STREAM ---
    # Trigger availableNow=True makes the stream behave like a batch job (process all new files then stop)
    query = (final_df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true") # Allow the Delta table to adapt to new JSON fields
        .trigger(availableNow=True)
        .toTable(target_table))
    
    query.awaitTermination()
    
    # Final Post-Ingestion Validation
    new_count = spark.table(target_table).filter(F.col("source") == "chunk3.json").count()
    print(f"   - Successfully processed. Current rows from chunk3: {new_count}")

# --- 5. ORCHESTRATION ---
def run_json_ingestion_pipeline():
    """
    Main loop that parses dataset names from a widget and triggers the ingestion.
    """
    datasets_json = dbutils.widgets.get("datasets_json")
    dataset_list = json.loads(datasets_json)
    
    for ds in dataset_list:
        clean_name = ds.lower()
        # Define paths for both Spark (dbfs:/) and local file system checks
        vol_path_spark = f"{VOLUME_PREFIX}/{clean_name}"
        vol_path_check = f"/Volumes/data_landing/data_raw/{clean_name}"
        full_table_name = f"{DEST_CATALOG}.{DEST_SCHEMA}.{clean_name}"

        try:
            # Pre-check: Verify the 'chunks' directory and target file actually exist
            files = dbutils.fs.ls(f"{vol_path_check}/chunks")
            if any(f.name == "chunk3.json" for f in files):
                print(f"\n[PROCESSING] {ds}")
                process_json_stream(ds, vol_path_spark, full_table_name)
            else:
                print(f"\n[SKIP] {ds}: chunk3.json not found in chunks folder.")
        except Exception as e:
            if "java.io.FileNotFoundException" in str(e):
                 print(f"\n[SKIP] {ds}: 'chunks' folder hierarchy not found.")
            else:
                 print(f"\n[ERROR] {ds}: {str(e)}")

if __name__ == "__main__":
    run_json_ingestion_pipeline()

**Unit testing:**

The validation script performs a battery of four critical checks for each dataset passed via the datasets_json widget:
- Table Existence: Confirms the Delta table was successfully initialized in the data_bronze.bronze schema.
- Source Specificity: Validates that records originating specifically from chunk3.json are present (proving the pathGlobFilter worked).
- Schema Enforcement: Ensures the "All-String" rule is maintained (all data columns must be StringType).
- Audit Integrity: Confirms load_dt is a valid TimestampType and not a string, allowing for future temporal analysis in the Silver layer.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, TimestampType
import json

def validate_json_ingestion(dataset_name):
    """
    Performs a technical audit of the 'chunk3.json' segment for a specific dataset.
    This function verifies row presence and schema compliance within the Bronze tier.
    """
    # Construct the target table path using the standard Bronze hierarchy
    table_name = f"{DEST_CATALOG}.{DEST_SCHEMA}.{dataset_name.lower()}"
    
    # Initialize a reporting object to track results for the final summary table
    report = {
        "Dataset": dataset_name,
        "Table": table_name,
        "Status": "SKIPPED", # Default status
        "JSON_Rows": 0,
        "Issues": []
    }

    # 1. EXISTENCE CHECK: Verify the Delta table was initialized in the Catalog
    if not spark.catalog.tableExists(table_name):
        report["Issues"].append("Table does not exist in Catalog.")
        return report

    try:
        # 2. SOURCE FILTERING: Isolate records specific to the JSON Auto Loader phase
        # This prevents CSV or XML records from skewing the JSON-specific validation
        json_df = spark.table(table_name).filter(F.col("source") == "chunk3.json")
        json_count = json_df.count()
        report["JSON_Rows"] = json_count

        # Check if the JSON file actually delivered any data to the table
        if json_count == 0:
            report["Issues"].append("No records found with source 'chunk3.json'.")
            report["Status"] = "NOT_FOUND"
            return report

        # 3. SCHEMA INTEGRITY: Enforce the 'Bronze String Standard'
        # We verify that no numeric or boolean types leaked in, ensuring raw data is preserved
        schema_errors = []
        for field in json_df.schema:
            # The system-generated load_dt must be a TimestampType
            if field.name == "load_dt":
                if not isinstance(field.dataType, TimestampType):
                    schema_errors.append(f"load_dt is {field.dataType}, expected Timestamp")
            # All other business data columns MUST be StringType in the Bronze tier
            else:
                if not isinstance(field.dataType, StringType):
                    schema_errors.append(f"{field.name} is {field.dataType}, expected String")
        
        # Consolidate findings into the final status
        if schema_errors:
            report["Issues"].extend(schema_errors)
            report["Status"] = "SCHEMA_FAIL"
        else:
            report["Status"] = "PASSED"

    except Exception as e:
        # Capture technical failures (e.g., table locks or permission errors)
        report["Status"] = "ERROR"
        report["Issues"].append(str(e)[:100])

    return report

# --- MAIN TEST ORCHESTRATION ---

# 1. FETCH PARAMETERS: Retrieve the list of datasets to validate from the notebook widget
datasets_val = dbutils.widgets.get("datasets_json")
target_list = json.loads(datasets_val)

# 2. BATCH VALIDATION: Iterate through the target datasets and collect audit results
all_reports = []
for ds in target_list:
    all_reports.append(validate_json_ingestion(ds))

# 3. REPORTING: Transform the results into a Spark DataFrame for visual analysis
report_df = spark.createDataFrame(all_reports)

print("--- BRONZE LAYER AUTO LOADER VALIDATION REPORT ---")
# Present the report with columns ordered for management review
display(report_df.select("Status", "Dataset", "JSON_Rows", "Issues", "Table"))

# 4. CRITICAL FAILURE ALERTING: Identify datasets that require manual intervention
# A 'SCHEMA_FAIL' or 'ERROR' will flag the pipeline for administrative review
failures = report_df.filter(F.col("Status").isin(["SCHEMA_FAIL", "ERROR"])).count()
if failures > 0:
    print(f"CRITICAL: {failures} datasets failed technical validation. Review the 'Issues' column.")